In [ ]:
# Build swimmer-only YOLO labels from COCO annotation JSON files
import json
from pathlib import Path

root = Path(r"D:\Python\Projects\Maritime-SAR")
data_dir = root / "data"
annotations_dir = data_dir / "annotations"

# Only keep these COCO category names (case-insensitive)
_class = {"swimmer"}

# Output YOLO class id for swimmer (aligned with NAMES in the next cell)
swimmer_class = 1

split_to_json = {
    "train": "instances_train.json",
    "val": "instances_val.json",
    "test": "instances_test.json",
}

def coco_bbox_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    x1 = max(0.0, float(x))
    y1 = max(0.0, float(y))
    x2 = min(float(img_w), x1 + max(0.0, float(w)))
    y2 = min(float(img_h), y1 + max(0.0, float(h)))

    bw = max(0.0, x2 - x1)
    bh = max(0.0, y2 - y1)
    if bw <= 0.0 or bh <= 0.0 or img_w <= 0 or img_h <= 0:
        return None

    xc = x1 + bw / 2.0
    yc = y1 + bh / 2.0
    return (
        xc / float(img_w),
        yc / float(img_h),
        bw / float(img_w),
        bh / float(img_h),
    )

for split, ann_filename in split_to_json.items():
    ann_path = annotations_dir / ann_filename
    images_dir = data_dir / split / "images"
    labels_dir = data_dir / split / "labels"
    labels_dir.mkdir(parents=True, exist_ok=True)

    if not ann_path.exists():
        print(f"[{split}] skip: {ann_path.name} not found")
        continue

    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    categories = coco.get("categories", [])
    images = coco.get("images", [])
    annotations = coco.get("annotations", [])

    swimmer_cat_ids = {
        int(cat["id"])
        for cat in categories
        if str(cat.get("name", "")).strip().lower() in _class
    }

    image_meta = {int(img["id"]): img for img in images}

    # Prepare empty label lists for all images listed in the JSON
    labels_by_stem = {}
    for img in images:
        stem = Path(str(img.get("file_name", ""))).stem
        if stem:
            labels_by_stem.setdefault(stem, [])

    kept = 0
    dropped = 0

    for ann in annotations:
        cat_id = int(ann.get("category_id", -1))
        if cat_id not in swimmer_cat_ids:
            dropped += 1
            continue

        img_id = int(ann.get("image_id", -1))
        img_info = image_meta.get(img_id)
        if img_info is None:
            dropped += 1
            continue

        file_name = str(img_info.get("file_name", ""))
        stem = Path(file_name).stem
        if not stem:
            dropped += 1
            continue

        img_w = int(img_info.get("width", 0))
        img_h = int(img_info.get("height", 0))
        bbox = ann.get("bbox", None)
        if bbox is None or len(bbox) != 4:
            dropped += 1
            continue

        yolo_box = coco_bbox_to_yolo(bbox, img_w, img_h)
        if yolo_box is None:
            dropped += 1
            continue

        xc, yc, w, h = yolo_box
        labels_by_stem.setdefault(stem, []).append(
            f"{swimmer_class} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
        )
        kept += 1

    # Write one label file per image in COCO JSON (empty files are valid for background images)
    written = 0
    for stem, lines in labels_by_stem.items():
        out_path = labels_dir / f"{stem}.txt"
        with open(out_path, "w", encoding="utf-8") as f:
            if lines:
                f.write("\n".join(lines) + "\n")
        written += 1

In [ ]:
import glob
from pathlib import Path
import cv2
import yaml
from tqdm import tqdm

root = Path(r"D:\Python\Projects\Maritime-SAR")  # project root
splits = ["train", "val", "test"]

# Contain annotions
data_dir = root / "data"

# Output sliced dataset
out_data_dir = root / "data_sahi"

# Slice params
slice_h = 640
slice_w = 640
overlap_ratio_h = 0.10
overlap_ratio_w = 0.10

min_ratio_tiles = 0.10   # keep box if at least 10% of original box area remains in tile
min_box_size = 2         # drop too tiny degenerate boxes

# class names
NAMES = {
    0: "ignored",
    1: "swimmer"
}

# don't apply sahi on a dataset
keep_tiles = {"test"}

def yolo_to_xyxy(xc, yc, w, h, img_w, img_h):
    x1 = (xc - w / 2.0) * img_w
    y1 = (yc - h / 2.0) * img_h
    x2 = (xc + w / 2.0) * img_w
    y2 = (yc + h / 2.0) * img_h
    return x1, y1, x2, y2

def xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h):
    bw = max(0.0, x2 - x1)
    bh = max(0.0, y2 - y1)
    xc = x1 + bw / 2.0
    yc = y1 + bh / 2.0
    return xc / img_w, yc / img_h, bw / img_w, bh / img_h

def clip_box_to_tile(box, tile):
    # box: (x1, y1, x2, y2) in full-image coords
    # tile: (tx1, ty1, tx2, ty2) in full-image coords
    x1, y1, x2, y2 = box
    tx1, ty1, tx2, ty2 = tile
    ix1 = max(x1, tx1)
    iy1 = max(y1, ty1)
    ix2 = min(x2, tx2)
    iy2 = min(y2, ty2)
    return ix1, iy1, ix2, iy2

def area_xyxy(b):
    x1, y1, x2, y2 = b
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def read_yolo_labels(label_path: Path):
    anns = []
    if not label_path.exists():
        return anns
    with open(label_path, "r", encoding="utf-8") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            p = ln.split()
            if len(p) != 5:
                continue
            cls = int(float(p[0]))
            xc, yc, w, h = map(float, p[1:])
            anns.append((cls, xc, yc, w, h))
    return anns

def generate_slices(img_h, img_w, slice_h, slice_w, overlap_ratio_h, overlap_ratio_w):
    step_h = max(1, int(slice_h * (1 - overlap_ratio_h)))
    step_w = max(1, int(slice_w * (1 - overlap_ratio_w)))

    ys = list(range(0, max(1, img_h - slice_h + 1), step_h))
    xs = list(range(0, max(1, img_w - slice_w + 1), step_w))

    ys.append(max(0, img_h - slice_h))
    xs.append(max(0, img_w - slice_w))

    # remove dup
    ys = sorted(set(ys))
    xs = sorted(set(xs))

    return [(x, y, min(x + slice_w, img_w), min(y + slice_h, img_h)) for y in ys for x in xs]

def process_split(split: str):
    src_img_dir = data_dir / split / "images"
    src_lbl_dir = data_dir / split / "labels"

    out_img_dir = out_data_dir / split / "images"
    out_lbl_dir = out_data_dir / split / "labels"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    image_paths = sorted(glob.glob(str(src_img_dir / "*.jpg")))
    image_paths += sorted(glob.glob(str(src_img_dir / "*.jpeg")))
    image_paths += sorted(glob.glob(str(src_img_dir / "*.png")))

    total_tiles = 0
    total_kept_boxes = 0

    for img_path in tqdm(image_paths, desc=f"Slicing {split}"):
        img_path = Path(img_path)
        stem = img_path.stem
        label_path = src_lbl_dir / f"{stem}.txt"

        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_h, img_w = img.shape[:2]

        anns = read_yolo_labels(label_path)

        # convert all GT boxes to full-image xyxy once
        full_boxes = []
        for cls, xc, yc, w, h in anns:
            x1, y1, x2, y2 = yolo_to_xyxy(xc, yc, w, h, img_w, img_h)
            full_boxes.append((cls, (x1, y1, x2, y2)))

        tiles = generate_slices(
            img_h, img_w,
            slice_h, slice_w,
            overlap_ratio_h, overlap_ratio_w
        )

        for t_idx, (tx1, ty1, tx2, ty2) in enumerate(tiles):
            tile_img = img[ty1:ty2, tx1:tx2]
            th, tw = tile_img.shape[:2]
            if th <= 0 or tw <= 0:
                continue

            out_stem = f"{stem}__x{tx1}_y{ty1}_w{tw}_h{th}"
            out_img_path = out_img_dir / f"{out_stem}.jpg"
            out_lbl_path = out_lbl_dir / f"{out_stem}.txt"

            tile_lines = []
            for cls, box in full_boxes:
                inter = clip_box_to_tile(box, (tx1, ty1, tx2, ty2))
                inter_area = area_xyxy(inter)
                orig_area = area_xyxy(box)

                if orig_area <= 0:
                    continue

                # keep if enough of object remains in tile
                if inter_area / orig_area < min_ratio_tiles:
                    continue

                ix1, iy1, ix2, iy2 = inter
                bw = ix2 - ix1
                bh = iy2 - iy1
                if bw < min_box_size or bh < min_box_size:
                    continue

                # move to tile-local coordinates
                lx1, ly1 = ix1 - tx1, iy1 - ty1
                lx2, ly2 = ix2 - tx1, iy2 - ty1

                xc_n, yc_n, w_n, h_n = xyxy_to_yolo(lx1, ly1, lx2, ly2, tw, th)

                # clamp for safety
                xc_n = min(max(xc_n, 0.0), 1.0)
                yc_n = min(max(yc_n, 0.0), 1.0)
                w_n  = min(max(w_n,  0.0), 1.0)
                h_n  = min(max(h_n,  0.0), 1.0)

                tile_lines.append(f"{cls} {xc_n:.6f} {yc_n:.6f} {w_n:.6f} {h_n:.6f}")


            if (len(tile_lines) == 0) and (split not in keep_tiles):
                continue

            # Save tile image always (keeps backgrounds too)
            cv2.imwrite(str(out_img_path), tile_img)

            # Write label file (empty file allowed for background tiles)
            with open(out_lbl_path, "w", encoding="utf-8") as f:
                if tile_lines:
                    f.write("\n".join(tile_lines) + "\n")

            total_tiles += 1
            total_kept_boxes += len(tile_lines)




# Run slicing for train/val/test
for split in splits:
    process_split(split)

# Write yolo data yaml for sliced dataset
data_yaml = {
    "path": str(out_data_dir.resolve()),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": NAMES
}

yaml_path = root / "data_sahi.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

In [17]:
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Force PyTorch to use GPU
torch.cuda.set_device(0)
device = torch.device('cuda:0')

GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.59 GB


In [2]:
from ultralytics import YOLO

# Load YOLOv8n-p2 model (P2 head for better small object detection)
# P2 adds an extra detection head at 1/4 resolution for detecting smaller objects
model = YOLO('yolov8-p2.yaml')
model.info()

WARNING no model scale passed. Assuming scale='n'.
YOLOv8-p2 summary: 161 layers, 3,354,144 parameters, 3,354,128 gradients, 17.4 GFLOPs


(161, 3354144, 3354128, 17.4345728)

In [ ]:
# Train the model on GPU with P2 head
results = model.train(
    data=r'D:\Python\Projects\Community\School\DAT\SeaDroneSee\data\data_sahi.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device='cuda:0',
    workers=8,
    patience=20,
    save=True,
    project='runs',
    name='seadronessee_yolov8n_p2',
    exist_ok=True,
    pretrained=True,
    optimizer='auto',  # Auto = SGD
    verbose=True,
    seed=42,
    val=True,
    plots=True,
    cache=False  # Set this to true if you have a lot of RAM, 9GB of free RAM required
)

New https://pypi.org/project/ultralytics/8.4.24 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bbox_loss=ciou, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Python\Projects\Community\School\DAT\SeaDroneSee\data\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yol

KeyboardInterrupt: 

In [ ]:
# Evaluate the model on validation set
metrics = model.val()

# Mean metrics across classes
precision = float(metrics.box.p.mean())
recall = float(metrics.box.r.mean())
f1 = 2 * precision * recall / (precision + recall + 1e-16)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")